In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

path = "../Graph RAG Dataset/testset1/"
loader = DirectoryLoader(path, glob="**/*.pdf", use_multithreading=True, max_concurrency=32, show_progress=True, loader_cls=PyPDFLoader)
docs = loader.load()

In [ ]:
personas = [
    # NEWS DOMESTIC / WORLD
    {
        "name": "Redaktor Depesz Zagranicznych",
        "role_description": "Dziennikarz analizujący wpływ wydarzeń międzynarodowych na bezpieczeństwo i gospodarkę Polski",
        "goals": [
            "Ocena skutków konfliktów geopolitycznych dla rynku energii w Polsce",
            "Analiza reakcji UE i NATO na napięcia globalne",
            "Weryfikacja wiarygodności źródeł zagranicznych"
        ]
    },
    {
        "name": "Analityk Bezpieczeństwa",
        "role_description": "Ekspert ds. bezpieczeństwa państwa i infrastruktury krytycznej",
        "goals": [
            "Identyfikacja zagrożeń hybrydowych dla Polski",
            "Analiza wpływu cyberataków na sektor finansowy",
            "Ocena stabilności regionu Europy Środkowo-Wschodniej"
        ]
    },

    # ECONOMY AND FINANCE
    {
        "name": "Analityk Makroekonomiczny",
        "role_description": "Specjalista ds. makroekonomii monitorujący inflację, PKB i politykę monetarną",
        "goals": [
            "Ocena wpływu decyzji banków centralnych na kurs złotego",
            "Analiza dynamiki inflacji w Polsce i UE",
            "Prognozowanie skutków zmian stóp procentowych dla kredytobiorców"
        ]
    },
    {
        "name": "Zarządzający Funduszem",
        "role_description": "Manager funduszu inwestycyjnego koncentrujący się na rynkach Europy Środkowej",
        "goals": [
            "Ocena ryzyka inwestycyjnego w sektorze bankowym",
            "Analiza kondycji spółek notowanych na GPW",
            "Monitorowanie przepływów kapitału zagranicznego"
        ]
    },

    # POLITICS
    {
        "name": "Komentator Polityczny",
        "role_description": "Publicysta analizujący strategie partii politycznych i skutki legislacyjne",
        "goals": [
            "Ocena wpływu nowych ustaw na przedsiębiorców",
            "Analiza stabilności koalicji rządowej",
            "Porównanie programów gospodarczych głównych partii"
        ]
    },
    {
        "name": "Doradca Legislacyjny",
        "role_description": "Ekspert ds. procesu legislacyjnego monitorujący projekty ustaw",
        "goals": [
            "Identyfikacja luk prawnych w projektach ustaw",
            "Analiza skutków regulacji dla sektora MŚP",
            "Ocena zgodności nowych przepisów z prawem UE"
        ]
    },

    # POLISH EXPORT
    {
        "name": "Dyrektor Eksportu",
        "role_description": "Manager w firmie produkcyjnej odpowiedzialny za ekspansję zagraniczną",
        "goals": [
            "Analiza barier celnych i regulacyjnych na rynkach UE i poza UE",
            "Ocena wpływu kursów walut na marżę eksportową",
            "Identyfikacja nowych rynków zbytu dla polskich produktów"
        ]
    },
    {
        "name": "Analityk Handlu Zagranicznego",
        "role_description": "Ekspert badający dane GUS i Eurostatu dotyczące wymiany handlowej",
        "goals": [
            "Analiza struktury polskiego eksportu według branż",
            "Ocena zależności eksportowych Polski od Niemiec i Chin",
            "Monitorowanie bilansu handlowego i trendów długoterminowych"
        ]
    }
]

In [ ]:
def build_single_hop_prompt(persona, chunk):
    return f"""
Jesteś: {persona['name']}
Rola: {persona['role_description']}
Cele: {', '.join(persona['goals'])}

Na podstawie poniższego fragmentu artykułu wygeneruj 7 konkretnych pytań,
na które odpowiedź znajduje się wyłącznie w tym tekście.

Pytania mają być:
- precyzyjne
- faktograficzne
- możliwe do jednoznacznej weryfikacji

Tekst:
{chunk}
"""

In [ ]:
def build_multihop_prompt(persona, chunk_a, chunk_b):
    return f"""
Jesteś: {persona['name']}
Rola: {persona['role_description']}
Cele: {', '.join(persona['goals'])}

Masz dwa fragmenty tego samego artykułu.

Wygeneruj 3 pytania wymagające połączenia informacji z OBU fragmentów.
Odpowiedź nie może wynikać wyłącznie z jednego z nich.

Fragment 1:
{chunk_a}

Fragment 2:
{chunk_b}
"""

In [ ]:
def build_cross_doc_multihop_prompt(persona, chunk_a, chunk_b):
    return f"""
Jesteś: {persona['name']}
Rola: {persona['role_description']}
Cele: {', '.join(persona['goals'])}

Masz fragmenty z dwóch różnych artykułów prasowych.

Wygeneruj 3 pytania analityczne, które wymagają:
- porównania
- syntezy
- wskazania zależności między tekstami

Fragment artykułu A:
{chunk_a}

Fragment artykułu B:
{chunk_b}
"""

In [ ]:
def build_general_prompt(persona, article_text):
    return f"""
Jesteś: {persona['name']}
Rola: {persona['role_description']}
Cele: {', '.join(persona['goals'])}

Na podstawie artykułu wygeneruj 5 ogólnych pytań analitycznych,
które badają szersze konsekwencje opisywanych zjawisk.

Pytania mają:
- dotyczyć trendów
- uwzględniać kontekst gospodarczy lub regulacyjny
- wymagać rozumowania

Artykuł:
{article_text}
"""

In [ ]:
for doc in docs:
    text = doc.page_content

In [ ]:
def call_llm(prompt, temperature=0.7):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Generujesz pytania testowe do systemu RAG. Nie twórz odpowiedzi, tylko pytania."},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

In [ ]:
from collections import defaultdict

articles = defaultdict(list)

for doc in docs:
    articles[doc.metadata["source"]].append(doc.page_content)

In [ ]:
def generate_intra_article_multihop(persona, pages):
    if len(pages) < 2:
        return None

    prompt = build_multihop_prompt(
        persona,
        pages[0],
        pages[1]
    )

    return call_llm(prompt)

In [ ]:
import random

def generate_cross_doc_multihop(persona, articles):
    keys = list(articles.keys())
    a, b = random.sample(keys, 2)

    prompt = build_cross_doc_multihop_prompt(
        persona,
        articles[a][0],
        articles[b][0]
    )

    return call_llm(prompt)

In [ ]:
def get_full_article(pages):
    return "\n\n".join(pages)

In [ ]:
full_text = get_full_article(articles[source])
prompt = build_general_prompt(persona, full_text)